# Advanced Citation & Text Highlighting System for Document RAG

## Overview
Implement NotebookLM-style features:
- **Citations with Hyperlinks**: Clickable references to source documents and page numbers
- **Text Highlighting**: Select passages in documents for analysis
- **Smart Explanations**: AI-powered explanations of highlighted text within document context

This notebook demonstrates the architecture and implementation patterns.

## 1. Set Up Citation Management Tools

Define TypeScript interfaces and data structures for managing citations with document references and metadata.

In [ ]:
# TypeScript Citation Management Interfaces
citation_interfaces = """
// src/types/Citation.ts
export interface Citation {
  id: string;                    // Unique citation ID
  quotedText: string;           // The actual quoted text
  sourceDocument: string;       // Document name (e.g., "report.pdf")
  pageNumber: number;           // Page where quote appears
  chunkIndex: number;           // Index in vector store
  startOffset: number;          // Character position in chunk
  endOffset: number;            // Character position end
  context: string;              // Surrounding context (50 chars before/after)
  timestamp: Date;
  citationFormat: 'APA' | 'MLA' | 'Chicago'; // Citation format
}

export interface CitationLink {
  text: string;                 // Display text for hyperlink
  documentPath: string;         // Path/hash to document
  pageNumber: number;
  jumpToId?: string;            // Element ID to scroll to
  isClickable: boolean;
}

export interface HighlightedText {
  id: string;
  text: string;
  source: Citation;
  explanation?: string;         // AI-generated explanation
  tags?: string[];              // User-defined tags
  createdAt: Date;
}

export interface CitationReport {
  title: string;
  documentName: string;
  citations: Citation[];
  highlights: HighlightedText[];
  totalCount: number;
  generatedAt: Date;
}
"""

print("Citation Management Interfaces:")
print(citation_interfaces)

## 2. Extract and Manage Source Citations

Create a CitationService class to parse documents, extract quoted passages with metadata, and maintain a citation index.

In [ ]:
# CitationService Implementation
citation_service = """
// src/services/CitationService.ts
import { Citation, HighlightedText, CitationReport } from '../types/Citation';

export class CitationService {
  private citations: Map<string, Citation> = new Map();
  private highlights: Map<string, HighlightedText> = new Map();

  /**
   * Extract a citation from a chunk with metadata
   */
  createCitation(
    quotedText: string,
    sourceDocument: string,
    pageNumber: number,
    chunkIndex: number,
    context: string
  ): Citation {
    const citation: Citation = {
      id: `cite_${Date.now()}_${Math.random().toString(36).substr(2, 9)}`,
      quotedText: quotedText.trim(),
      sourceDocument,
      pageNumber,
      chunkIndex,
      startOffset: context.indexOf(quotedText),
      endOffset: context.indexOf(quotedText) + quotedText.length,
      context: context,
      timestamp: new Date(),
      citationFormat: 'APA'
    };
    
    this.citations.set(citation.id, citation);
    return citation;
  }

  /**
   * Find citation by quote text (for highlighting)
   */
  findCitationByText(text: string): Citation | undefined {
    return Array.from(this.citations.values()).find(
      c => c.quotedText.toLowerCase() === text.toLowerCase()
    );
  }

  /**
   * Create a highlighted text with optional explanation
   */
  createHighlight(
    text: string,
    citation: Citation,
    explanation?: string
  ): HighlightedText {
    const highlight: HighlightedText = {
      id: \`highlight_\${Date.now()}_\${Math.random().toString(36).substr(2, 9)}\`,
      text,
      source: citation,
      explanation,
      tags: [],
      createdAt: new Date()
    };
    
    this.highlights.set(highlight.id, highlight);
    return highlight;
  }

  /**
   * Format citation in different styles
   */
  formatCitation(citation: Citation, format: 'APA' | 'MLA' | 'Chicago' = 'APA'): string {
    const docName = citation.sourceDocument;
    const page = citation.pageNumber;
    
    switch (format) {
      case 'APA':
        return \`(\${docName}, p. \${page})\`;
      case 'MLA':
        return \`("\${citation.quotedText}" \${docName} \${page})\`;
      case 'Chicago':
        return \`\${docName}, page \${page}.\`;
      default:
        return \`\${docName}, p. \${page}\`;
    }
  }

  /**
   * Get all citations for a document
   */
  getCitationsForDocument(documentName: string): Citation[] {
    return Array.from(this.citations.values())
      .filter(c => c.sourceDocument === documentName);
  }

  /**
   * Generate citation report
   */
  generateReport(documentName: string): CitationReport {
    return {
      title: \`Citation Report: \${documentName}\`,
      documentName,
      citations: this.getCitationsForDocument(documentName),
      highlights: Array.from(this.highlights.values())
        .filter(h => h.source.sourceDocument === documentName),
      totalCount: this.citations.size,
      generatedAt: new Date()
    };
  }
}
"""

print("CitationService Implementation:")
print(citation_service)

## 3. Implement Text Highlighting System

Build React components that allow users to highlight and select text passages for deeper analysis.

In [ ]:
# Text Highlighting React Component
highlighting_component = """
// src/components/HighlightableText.tsx
import React, { useState } from 'react';
import { Citation } from '../types/Citation';
import { Highlighter, MessageSquare } from 'lucide-react';

interface HighlightableTextProps {
  text: string;
  citations: Citation[];
  onHighlight: (text: string, citation: Citation) => void;
  onExplain: (text: string) => void;
}

export function HighlightableText({
  text,
  citations,
  onHighlight,
  onExplain
}: HighlightableTextProps) {
  const [selectedText, setSelectedText] = useState('');
  const [selectionPosition, setSelectionPosition] = useState<{x: number; y: number} | null>(null);

  const handleTextSelection = (e: React.MouseEvent) => {
    const selection = window.getSelection();
    if (selection && selection.toString().length > 0) {
      const text = selection.toString();
      setSelectedText(text);
      
      // Find citation for this text
      const citation = citations.find(c => 
        c.quotedText.includes(text) || text.includes(c.quotedText)
      );
      
      if (citation) {
        // Position popup near selection
        const range = selection.getRangeAt(0);
        const rect = range.getBoundingClientRect();
        setSelectionPosition({
          x: rect.left + rect.width / 2,
          y: rect.top - 10
        });
      }
    }
  };

  const handleHighlight = (text: string) => {
    const citation = citations.find(c => c.quotedText.includes(text));
    if (citation) {
      onHighlight(text, citation);
    }
  };

  return (
    <div
      onMouseUp={handleTextSelection}
      className="prose prose-sm max-w-none p-4 bg-white rounded-lg border border-gray-200"
    >
      {/* Render text with citation marks */}
      {renderHighlightedText(text, citations)}

      {/* Selection Toolbar */}
      {selectedText && selectionPosition && (
        <div
          className="fixed bg-white shadow-lg rounded-lg p-2 flex gap-2 z-50"
          style={{
            left: \`\${selectionPosition.x}px\`,
            top: \`\${selectionPosition.y}px\`,
            transform: 'translate(-50%, -100%)'
          }}
        >
          <button
            onClick={() => handleHighlight(selectedText)}
            className="flex items-center gap-1 px-3 py-1 text-sm hover:bg-yellow-100 rounded"
            title="Highlight this text"
          >
            <Highlighter size={16} />
            Highlight
          </button>
          <button
            onClick={() => onExplain(selectedText)}
            className="flex items-center gap-1 px-3 py-1 text-sm hover:bg-blue-100 rounded"
            title="Explain this text"
          >
            <MessageSquare size={16} />
            Explain
          </button>
        </div>
      )}
    </div>
  );
}

/**
 * Helper: Render text with citation highlights
 */
function renderHighlightedText(text: string, citations: Citation[]) {
  let lastIndex = 0;
  const elements = [];

  citations.forEach((citation, idx) => {
    const quoteIndex = text.indexOf(citation.quotedText);
    if (quoteIndex === -1) return;

    // Add text before quote
    if (quoteIndex > lastIndex) {
      elements.push(
        <span key={\`text_\${idx}\`}>
          {text.substring(lastIndex, quoteIndex)}
        </span>
      );
    }

    // Add highlighted quote
    elements.push(
      <mark
        key={\`quote_\${idx}\`}
        className="bg-yellow-200 cursor-pointer hover:bg-yellow-300 transition-colors px-0.5 rounded"
        title={\`\${citation.sourceDocument}, p. \${citation.pageNumber}\`}
      >
        {citation.quotedText}
      </mark>
    );

    lastIndex = quoteIndex + citation.quotedText.length;
  });

  // Add remaining text
  if (lastIndex < text.length) {
    elements.push(
      <span key="text_end">
        {text.substring(lastIndex)}
      </span>
    );
  }

  return <>{elements}</>;
}
"""

print("Text Highlighting Component:")
print(highlighting_component)

## 4. Create Interactive Explanation Queries

Implement AI-powered explanations for selected text that stay grounded in document context.

In [ ]:
# Explanation Service Implementation
explanation_service = """
// src/services/ExplanationService.ts
import { GeminiService } from './GeminiService';
import { VectorStore } from './VectorStore';

export class ExplanationService {
  constructor(
    private gemini: GeminiService,
    private vectorStore: VectorStore
  ) {}

  /**
   * Explain a highlighted text passage within document context
   */
  async explainHighlightedText(
    highlightedText: string,
    documentName: string,
    pageNumber?: number
  ): Promise<string> {
    try {
      // Find surrounding context
      const entries = this.vectorStore.search(
        await this.gemini.embed(highlightedText),
        k: 3
      );

      if (!entries.length) {
        throw new Error('No context found for highlighted text');
      }

      // Build context with surrounding passages
      const contextChunks = entries
        .map(e => e.text)
        .join('\\n\\n');

      const prompt = \`
You are a helpful assistant that explains text passages within their document context.

HIGHLIGHTED TEXT:
"\${highlightedText}"

DOCUMENT CONTEXT:
\${contextChunks}

${pageNumber ? \`This text appears on page \${pageNumber}.\` : ''}

TASK:
Provide a clear, concise explanation of what this highlighted text means in the context 
of the document. Include:
1. What the passage means
2. Why it's important in this context
3. How it relates to surrounding information
4. Any key concepts it introduces

Keep your explanation grounded ONLY in the provided document context.
Answer in the same language as the document.
      \`;

      return await this.gemini.generateResponse(
        prompt,
        contextChunks,
        [],
        'explanation'
      );
    } catch (error) {
      console.error('Error explaining text:', error);
      throw new Error('Failed to explain the highlighted text');
    }
  }

  /**
   * Explain a concept with multiple examples from document
   */
  async explainConceptWithExamples(
    concept: string,
    maxExamples: number = 3
  ): Promise<{explanation: string; examples: string[]}> {
    try {
      // Find multiple relevant passages
      const conceptVector = await this.gemini.embed(concept);
      const entries = this.vectorStore.search(conceptVector, k: maxExamples + 2);

      const examples = entries
        .slice(0, maxExamples)
        .map(e => e.text);

      const contextChunks = entries.map(e => e.text).join('\\n\\n');

      const prompt = \`
Explain the concept "\${concept}" using examples from the provided document.

CONCEPT: \${concept}

DOCUMENT PASSAGES:
\${contextChunks}

Provide:
1. A clear definition of the concept
2. Key characteristics
3. How it's used in the document
4. Any important distinctions or nuances

Use ONLY the document content for your explanation.
      \`;

      const explanation = await this.gemini.generateResponse(
        prompt,
        contextChunks,
        [],
        'concept'
      );

      return {
        explanation,
        examples
      };
    } catch (error) {
      console.error('Error explaining concept:', error);
      throw new Error('Failed to explain the concept');
    }
  }

  /**
   * Generate study notes for highlighted passage
   */
  async generateStudyNotes(
    highlightedText: string,
    noteType: 'summary' | 'questions' | 'key_terms' = 'summary'
  ): Promise<string> {
    const prompt = \`
Create study notes for this highlighted text.
Type: \${noteType}

TEXT: "\${highlightedText}"

${noteType === 'summary' ? 'Summarize in bullet points' : ''}
${noteType === 'questions' ? 'Generate study questions' : ''}
${noteType === 'key_terms' ? 'Extract and explain key terms' : ''}
    \`;

    return await this.gemini.generateResponse(
      prompt,
      highlightedText,
      [],
      'study_note'
    );
  }
}
"""

print("Explanation Service:")
print(explanation_service)

## 5. Build Hyperlink References to Documents

Create clickable citation links that navigate directly to source documents and page numbers.

In [ ]:
# Citation Hyperlink Components
hyperlink_components = """
// src/components/CitationLink.tsx
import React from 'react';
import { ExternalLink, FileText } from 'lucide-react';
import { Citation } from '../types/Citation';

interface CitationLinkProps {
  citation: Citation;
  onClick?: (citation: Citation) => void;
  format?: 'inline' | 'footnote' | 'popup';
}

export function CitationLink({
  citation,
  onClick,
  format = 'inline'
}: CitationLinkProps) {
  const handleClick = () => {
    onClick?.(citation);
    
    // Emit event to scroll document viewer to page
    window.dispatchEvent(new CustomEvent('citation-click', {
      detail: {
        documentName: citation.sourceDocument,
        pageNumber: citation.pageNumber,
        chunkIndex: citation.chunkIndex
      }
    }));
  };

  const formats = {
    inline: (
      <button
        onClick={handleClick}
        className="inline-text-blue-600 hover:text-blue-800 underline cursor-pointer text-sm transition-colors"
        title={\`Click to view: \${citation.sourceDocument}, page \${citation.pageNumber}\`}
      >
        {citation.sourceDocument} (p. {citation.pageNumber})
      </button>
    ),

    footnote: (
      <span className="text-xs text-gray-500 ml-1">
        <sup>
          <button
            onClick={handleClick}
            className="text-blue-600 hover:text-blue-800 underline"
            title={citation.sourceDocument}
          >
            [{citation.pageNumber}]
          </button>
        </sup>
      </span>
    ),

    popup: (
      <div className="bg-blue-50 border-l-4 border-blue-400 p-3 rounded inline-block">
        <div className="flex items-start gap-2">
          <FileText size={16} className="text-blue-600 flex-shrink-0 mt-0.5" />
          <div className="flex-1">
            <p className="text-sm font-semibold text-gray-800">
              {citation.sourceDocument}
            </p>
            <p className="text-xs text-gray-600">Page {citation.pageNumber}</p>
            <p className="text-xs text-gray-700 mt-1 italic">
              "{citation.quotedText.substring(0, 100)}..."
            </p>
            <button
              onClick={handleClick}
              className="text-xs text-blue-600 hover:text-blue-800 mt-2 flex items-center gap-1"
            >
              <ExternalLink size={12} />
              View in document
            </button>
          </div>
        </div>
      </div>
    )
  };

  return <>{formats[format]}</>;
}

// src/components/CitationBlock.tsx
/**
 * Display citation with full context and visual design
 */
export function CitationBlock({ citation }: { citation: Citation }) {
  return (
    <div className="my-4 p-4 bg-amber-50 border-l-4 border-amber-400 rounded-r-lg">
      <div className="flex items-start justify-between gap-4">
        <div className="flex-1">
          <p className="text-sm text-gray-600 mb-2">
            <strong>From:</strong> {citation.sourceDocument}, page {citation.pageNumber}
          </p>
          <blockquote className="text-base text-gray-800 italic border-l-4 border-amber-200 pl-3 py-2">
            "{citation.quotedText}"
          </blockquote>
          <p className="text-xs text-gray-500 mt-2">
            {citation.citationFormat}: {formatCitation(citation)}
          </p>
        </div>
        <button
          className="px-3 py-2 text-sm bg-white hover:bg-amber-100 border border-amber-200 rounded transition-colors flex items-center gap-2"
          onClick={() => window.dispatchEvent(new CustomEvent('citation-click', {
            detail: {
              documentName: citation.sourceDocument,
              pageNumber: citation.pageNumber
            }
          }))}
        >
          <ExternalLink size={14} />
          Open
        </button>
      </div>
    </div>
  );
}

/**
 * Format citation text
 */
function formatCitation(citation: Citation): string {
  switch (citation.citationFormat) {
    case 'APA':
      return \`(Author, p. \${citation.pageNumber})\`;
    case 'MLA':
      return \`("\${citation.quotedText.substring(0, 50)}..." \${citation.pageNumber})\`;
    case 'Chicago':
      return \`\${citation.sourceDocument}, p. \${citation.pageNumber}.\`;
    default:
      return \`\${citation.sourceDocument}, p. \${citation.pageNumber}\`;
  }
}
"""

print("Citation Hyperlink Components:")
print(hyperlink_components)

## 6. Generate Citation Reports

Compile all citations, highlights, and explanations into formatted reports with source attribution.

In [ ]:
# Citation Report Generation
report_generation = """
// src/services/ReportGenerator.ts
import { Citation, HighlightedText, CitationReport } from '../types/Citation';
import { CitationService } from './CitationService';

export class ReportGenerator {
  constructor(private citationService: CitationService) {}

  /**
   * Generate a comprehensive citation report
   */
  generateReport(
    documentName: string,
    format: 'markdown' | 'html' | 'text' = 'markdown'
  ): string {
    const report = this.citationService.generateReport(documentName);
    
    switch (format) {
      case 'markdown':
        return this.generateMarkdownReport(report);
      case 'html':
        return this.generateHtmlReport(report);
      case 'text':
        return this.generateTextReport(report);
      default:
        return this.generateMarkdownReport(report);
    }
  }

  /**
   * Generate markdown format report
   */
  private generateMarkdownReport(report: CitationReport): string {
    let md = \`# \${report.title}\\n\\n\`;
    md += \`**Generated:** \${report.generatedAt.toLocaleDateString()}\\n\\n\`;

    // Citations section
    md += \`## Citations (Total: \${report.citations.length})\\n\\n\`;
    report.citations.forEach((citation, idx) => {
      md += \`### \${idx + 1}. Page \${citation.pageNumber}\\n\`;
      md += \`> "\${citation.quotedText}"\\n\\n\`;
      md += \`**Format:**\\n\`;
      md += \`- APA: \${this.citationService.formatCitation(citation, 'APA')}\\n\`;
      md += \`- MLA: \${this.citationService.formatCitation(citation, 'MLA')}\\n\`;
      md += \`- Chicago: \${this.citationService.formatCitation(citation, 'Chicago')}\\n\\n\`;
    });

    // Highlights section
    md += \`## Highlights (Total: \${report.highlights.length})\\n\\n\`;
    report.highlights.forEach((highlight, idx) => {
      md += \`### \${idx + 1}. \${highlight.text.substring(0, 50)}...\\n\`;
      if (highlight.explanation) {
        md += \`**Explanation:**\\n\${highlight.explanation}\\n\\n\`;
      }
      if (highlight.tags && highlight.tags.length > 0) {
        md += \`**Tags:** \${highlight.tags.join(', ')}\\n\\n\`;
      }
    });

    return md;
  }

  /**
   * Generate HTML format report
   */
  private generateHtmlReport(report: CitationReport): string {
    let html = \`
    <html>
      <head>
        <title>\${report.title}</title>
        <style>
          body { font-family: Arial, sans-serif; line-height: 1.6; color: #333; }
          .citation { margin: 20px 0; padding: 15px; background: #f5f5f5; border-left: 4px solid #007bff; }
          .highlight { margin: 20px 0; padding: 15px; background: #fff3cd; border-left: 4px solid #ffc107; }
          blockquote { font-style: italic; margin-left: 20px; border-left: 3px solid #ccc; padding-left: 10px; }
          .meta { font-size: 0.9em; color: #666; }
        </style>
      </head>
      <body>
        <h1>\${report.title}</h1>
        <p class="meta">Generated: \${report.generatedAt.toLocaleString()}</p>
    \`;

    // Citations
    html += \`<h2>Citations (\${report.citations.length})</h2>\`;
    report.citations.forEach((citation, idx) => {
      html += \`
        <div class="citation">
          <h3>Citation \${idx + 1}</h3>
          <p><strong>Source:</strong> \${citation.sourceDocument}, Page \${citation.pageNumber}</p>
          <blockquote>\${citation.quotedText}</blockquote>
          <p class="meta">
            APA: \${this.citationService.formatCitation(citation, 'APA')}<br>
            MLA: \${this.citationService.formatCitation(citation, 'MLA')}<br>
            Chicago: \${this.citationService.formatCitation(citation, 'Chicago')}
          </p>
        </div>
      \`;
    });

    // Highlights
    html += \`<h2>Highlights (\${report.highlights.length})</h2>\`;
    report.highlights.forEach((highlight, idx) => {
      html += \`
        <div class="highlight">
          <h3>Highlight \${idx + 1}</h3>
          <blockquote>\${highlight.text}</blockquote>
          \${highlight.explanation ? \`<p><strong>Explanation:</strong> \${highlight.explanation}</p>\` : ''}
          \${highlight.tags && highlight.tags.length > 0 ? \`<p><strong>Tags:</strong> \${highlight.tags.join(', ')}</p>\` : ''}
        </div>
      \`;
    });

    html += \`
      </body>
    </html>
    \`;

    return html;
  }

  /**
   * Generate plain text format report
   */
  private generateTextReport(report: CitationReport): string {
    let text = \`\${report.title}\\n\`;
    text += \`\${'='.repeat(report.title.length)}\\n\\n\`;
    text += \`Generated: \${report.generatedAt.toLocaleString()}\\n\\n\`;

    text += \`CITATIONS (Total: \${report.citations.length})\\n\`;
    text += \`${'─'.repeat(40)}\\n\`;
    report.citations.forEach((citation, idx) => {
      text += \`\\n\${idx + 1}. PAGE \${citation.pageNumber}\\n\`;
      text += \`"\${citation.quotedText}"\\n\`;
      text += \`Source: \${citation.sourceDocument}\\n\`;
    });

    text += \`\\n\\nHIGHLIGHTS (Total: \${report.highlights.length})\\n\`;
    text += \`${'─'.repeat(40)}\\n\`;
    report.highlights.forEach((highlight, idx) => {
      text += \`\\n\${idx + 1}. \${highlight.text}\\n\`;
      if (highlight.explanation) text += \`   Explanation: \${highlight.explanation}\\n\`;
      if (highlight.tags) text += \`   Tags: \${highlight.tags.join(', ')}\\n\`;
    });

    return text;
  }

  /**
   * Export report to file
   */
  exportReport(
    documentName: string,
    format: 'markdown' | 'html' | 'text' | 'json' = 'markdown'
  ): Blob {
    const content = format === 'json' 
      ? JSON.stringify(this.citationService.generateReport(documentName), null, 2)
      : this.generateReport(documentName, format as 'markdown' | 'html' | 'text');

    const mimeType = {
      markdown: 'text/markdown',
      html: 'text/html',
      text: 'text/plain',
      json: 'application/json'
    }[format];

    return new Blob([content], { type: mimeType });
  }
}
"""

print("Citation Report Generation:")
print(report_generation)

## 7. Integration & Usage Examples

### Key Features Summary

| Feature | Implementation | Usage |
|---------|----------------|-------|
| **Citations** | CitationService | Auto-create from document chunks |
| **Text Selection** | HighlightableText component | Mouse selection with popup |
| **Explanations** | ExplanationService | AI-powered, grounded in context |
| **Hyperlinks** | CitationLink component | Click to jump to source |
| **Reports** | ReportGenerator | Export in MD/HTML/JSON/TXT |
| **Highlighting** | Visual marks on text | Persist selections with tags |

### Usage Flow

```
1. User uploads document
   ↓
2. Document processed → chunks created
   ↓
3. User views message with answer
   ↓
4. User highlights text
   ↓
5. Popup shows [Highlight] [Explain] buttons
   ↓
6a. Click [Highlight] → Mark with citation
   ↓
6b. Click [Explain] → Get AI explanation
   ↓
7. Citations are clickable → Jump to source page
   ↓
8. Export report with all citations
```

In [ ]:
# Practical Implementation Steps
implementation_guide = """
## Step 1: Create Citation Types & Services

1. Create src/types/Citation.ts with all interfaces
2. Create src/services/CitationService.ts for managing citations
3. Create src/services/ExplanationService.ts for AI explanations
4. Create src/services/ReportGenerator.ts for report generation

## Step 2: Add UI Components

1. Create src/components/HighlightableText.tsx
   - Handle text selection
   - Show popup toolbar on selection
   - Support highlighting and explain buttons

2. Create src/components/CitationLink.tsx
   - Display formatted citations
   - Support APA/MLA/Chicago formats
   - Clickable links to jump to source

3. Create src/components/CitationBlock.tsx
   - Display full citation with context
   - Visual design with borders
   - Open button for navigation

## Step 3: Integrate with ChatEngine

1. Add CitationService to ChatEngine
2. Add ExplanationService to ChatEngine
3. Add ReportGenerator to ChatEngine
4. Create new methods:
   - highlightText(text, citation)
   - explainHighlight(text)
   - generateCitationReport(format)

## Step 4: Update App.tsx

1. Import and use HighlightableText component
2. Add citation display in messages
3. Add buttons for:
   - Export Report
   - View All Citations
   - Study Highlights
4. Track selected highlights in state

## Step 5: Add to Message Response

Enhance ChatResponse type to include:
- citations: Citation[]
- highlightableContent: boolean
- fullContext: string  // For explanation service

## Testing Checklist

✓ Text selection works smoothly
✓ Popup shows with correct buttons
✓ Highlight creates citation
✓ Explain provides grounded content
✓ Citations are clickable
✓ Report generation works (all formats)
✓ No hallucinations in explanations
✓ Page numbers are accurate
✓ Multi-file citations work
✓ Export functionality works

## Performance Considerations

- Lazy load ExplanationService (on demand)
- Cache citations in IndexedDB
- Debounce text selection events
- Limit explanation context to top 5 chunks
- Use requestAnimationFrame for highlighting
"""

print("Implementation Strategy:")
print(implementation_guide)

In [ ]:
# ChatEngine Citation Integration
chatengine_integration = """
// Updated src/services/ChatEngine.ts

export interface ChatResponse {
  answer: string;
  sources: { text: string; pageNumber?: number; image?: string }[];
  citations: Citation[];  // NEW: Add citations to response
  highlightableContent: string;  // NEW: Content with citation markers
}

export class ChatEngine {
  // ... existing code ...
  
  private citationService: CitationService;
  private explanationService: ExplanationService;
  private reportGenerator: ReportGenerator;

  constructor() {
    // ... existing initialization ...
    this.citationService = new CitationService();
    this.explanationService = new ExplanationService(this.gemini, this.vectorStore);
    this.reportGenerator = new ReportGenerator(this.citationService);
  }

  /**
   * Enhanced query with citation tracking
   */
  async query(userQuery: string): Promise<ChatResponse> {
    const queryEmbedding = await this.gemini.embed(userQuery);
    const topChunks = this.vectorStore.search(queryEmbedding, 5);

    // Create citations for each source chunk
    const citations: Citation[] = topChunks.map((chunk, idx) => 
      this.citationService.createCitation(
        chunk.text.substring(0, 200),
        chunk.metadata.fileName,
        chunk.metadata.pageNumber || 1,
        chunk.metadata.chunkIndex,
        chunk.text
      )
    );

    // Generate response with context
    const answer = await this.gemini.generateResponse(userQuery, ...topChunks);

    return {
      answer,
      sources: topChunks.map(c => ({
        text: c.text,
        pageNumber: c.metadata.pageNumber,
        image: c.metadata.image
      })),
      citations,
      highlightableContent: answer  // Mark answer as highlightable
    };
  }

  /**
   * Explain highlighted text with full grounding
   */
  async explainHighlightedText(
    highlightedText: string,
    sourceDocument: string,
    pageNumber?: number
  ): Promise<string> {
    return await this.explanationService.explainHighlightedText(
      highlightedText,
      sourceDocument,
      pageNumber
    );
  }

  /**
   * Highlight text and create permanent reference
   */
  highlightText(text: string, citation: Citation): HighlightedText {
    return this.citationService.createHighlight(text, citation);
  }

  /**
   * Generate comprehensive citation report
   */
  generateCitationReport(format: 'markdown' | 'html' | 'text' = 'markdown'): string {
    const currentDoc = this.currentFile?.name || 'Document';
    return this.reportGenerator.generateReport(currentDoc, format);
  }

  /**
   * Export citations to file
   */
  exportCitationReport(format: 'markdown' | 'html' | 'json' = 'markdown'): Blob {
    const currentDoc = this.currentFile?.name || 'Document';
    return this.reportGenerator.exportReport(currentDoc, format);
  }

  /**
   * Get all highlights for current document
   */
  getAllHighlights(): HighlightedText[] {
    return this.citationService.getHighlightsForDocument(
      this.currentFile?.name || ''
    );
  }
}
"""

print("ChatEngine Integration:")
print(chatengine_integration)